In [1]:
# Scenario
# A student is building a simple AI assistant in Google Colab for a class project. The assistant is
# hosted on Azure Foundry, and the student wants to ask it questions like:

# “What can you help with?”

# “Explain this topic simply.”

# “Give me a short summary.”

# Instead of talking directly to the model through a chat app, the student uses Python code to connect to the Azure service.

In [3]:


from dotenv import load_dotenv
import os

# Load env variables
load_dotenv(dotenv_path=r"C:\Users\naina\Capgemini\Generative Ai\.env")

api_key = os.getenv("Foundary_api")
print("Loaded key:", api_key[:6] + "..." if api_key else "Missing")


Loaded key: BDaxmN...


In [8]:
from openai import OpenAI

endpoint = os.getenv("Foundary_endpoint")
my_agent = "email-assistant"
my_version = "2"

api_key = os.getenv("Foundary_api")

openai_client = OpenAI(
    base_url=f"{endpoint}/openai/v1",
    api_key=api_key
)

response = openai_client.responses.create(
    input=[{"role": "user", "content": "Tell me what you can help with."}],
    extra_body={
        "agent_reference": {
            "name": my_agent,
            "version": my_version,
            "type": "agent_reference"
        }
    },
)

print(response.output_text)

Certainly! As your professional email assistant, I can help you with:

- Drafting new formal emails for various business situations  
- Responding to business emails in a polite and effective manner  
- Proofreading and refining your existing emails for clarity and professionalism  
- Providing templates for common scenarios, such as meeting requests, follow-ups, thank you notes, or apologies  
- Ensuring your emails have the appropriate tone—whether gentle, assertive, or neutral—based on your needs  
- Suggesting subject lines and proper greetings/closings  
- Adapting your emails for international or cross-cultural communication  
- Answering email etiquette questions  

If you have a specific scenario or draft you’d like help with, just let me know!


In [9]:
from openai import OpenAI

endpoint = os.getenv("Foundary_endpoint")
my_agent = "Ai-CustomerSupport"
my_version = "2"

api_key = os.getenv("Foundary_api")

openai_client = OpenAI(
    base_url=f"{endpoint}/openai/v1",
    api_key=api_key
)

response = openai_client.responses.create(
    input=[{"role": "user", "content": "Tell me what you can help with."}],
    extra_body={
        "agent_reference": {
            "name": my_agent,
            "version": my_version,
            "type": "agent_reference"
        }
    },
)

print(response.output_text)

I can assist you with customer support inquiries such as orders, refunds, delivery status, and account-related issues. Please let me know how I can help you today!


In [12]:
from openai import OpenAI
import os

endpoint = os.getenv("Foundary_endpoint")
api_key = os.getenv("Foundary_api")

my_agent = "Ai-CustomerSupport"
my_version = "1"   

# 🔹 Initialize client
openai_client = OpenAI(
    base_url=f"{endpoint}/openai/v1",
    api_key=api_key
)

# 🔹 Prompt
user_prompt = """
Act as a professional AI Customer Support Assistant for an e-commerce platform.

Explain clearly what services you can provide, including:
1. Handling customer queries
2. Order tracking
3. Complaint resolution
4. Product recommendations

Also include a short example interaction with a customer.
"""

try:
    #  Call with Agent
    response = openai_client.responses.create(
        input=[
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        extra_body={
            "agent_reference": {
                "name": my_agent,
                "version": my_version,
                "type": "agent_reference"
            }
        },
    )

    print(" Agent Response:\n")
    print(response.output_text)

except Exception as e:
    print("⚠️ Agent failed, using fallback model...\n")
    print("Error:", e)

    # 🔹 Fallback (direct model call)
    response = openai_client.responses.create(
        model="gpt-4.1-mini",   # or your deployed model
        input=user_prompt
    )

    print("\n Fallback Response:\n")
    print(response.output_text)

 Agent Response:

As a professional AI Customer Support Assistant for an e-commerce platform, I offer a range of services to ensure a smooth and satisfying shopping experience for every customer. Here’s what I can do:

---

**1. Handling Customer Queries:**  
I am available 24/7 to answer any questions customers may have about products, services, policies, shipping, payments, or their accounts. My goal is to provide clear, accurate, and timely information.

**2. Order Tracking:**  
Customers can easily check the status of their orders with me. I can provide real-time updates on shipment progress, estimated delivery times, and tracking links.

**3. Complaint Resolution:**  
If a customer encounters an issue with their order—for example, damaged products, delayed deliveries, or payment problems—I will assist in gathering details, escalating to the right team, and following up to ensure a satisfactory resolution.

**4. Product Recommendations:**  
Based on customer preferences, browsing h

In [13]:
from openai import OpenAI
import os
import gradio as gr

# 🔹 Config
ENDPOINT = os.getenv("Foundary_endpoint")
API_KEY = os.getenv("Foundary_api")

AGENT_NAME = "Ai-CustomerSupport"
AGENT_VERSION = "1"   # change if needed
MODEL_NAME = "gpt-4.1-mini"

# 🔹 Initialize client
client = OpenAI(
    base_url=f"{ENDPOINT}/openai/v1",
    api_key=API_KEY
)

# 🔹 -------- GUARDRAILS --------
def apply_guardrails(user_input):
    blocked_words = ["hack", "fraud", "illegal", "attack"]

    for word in blocked_words:
        if word.lower() in user_input.lower():
            return False, "⚠️ Request blocked due to unsafe content."

    if len(user_input.strip()) == 0:
        return False, "⚠️ Please enter a valid query."

    return True, user_input


# 🔹 -------- CORE LOGIC --------
def get_response(user_input):
    # Apply guardrails
    is_safe, processed_input = apply_guardrails(user_input)
    if not is_safe:
        return processed_input

    prompt = f"""
    You are a professional AI Customer Support Assistant.

    User Query: {processed_input}

    Respond helpfully, politely, and clearly.
    """

    try:
        # Try Agent
        response = client.responses.create(
            input=prompt,
            extra_body={
                "agent_reference": {
                    "name": AGENT_NAME,
                    "version": AGENT_VERSION,
                    "type": "agent_reference"
                }
            }
        )
        return response.output_text

    except Exception:
        # Fallback Model
        response = client.responses.create(
            model=MODEL_NAME,
            input=prompt
        )
        return response.output_text


# 🔹 -------- GRADIO UI --------
iface = gr.Interface(
    fn=get_response,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Ask customer support something..."
    ),
    outputs="text",
    title="🛍️ AI Customer Support Assistant",
    description="Ask queries related to orders, complaints, products, etc."
)

# 🔹 Run app
if __name__ == "__main__":
    iface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
